# SHAP Cutoff Analysis

Extract quantitative transition points from SHAP dependence relationships.

**What this does:**
1. Loads OOF SHAP values from `shap_values_for_sharing.pkl`
2. Fits a LOESS smoother to the SHAP-vs-feature relationship for any feature
3. Finds **zero crossings** (where SHAP flips sign) with bootstrap confidence intervals
4. Finds **rate-of-change peaks** (steepest transitions, i.e. inflection regions)
5. Optionally stratifies by an interacting feature (e.g., `last.EJC`)
6. Produces a clean visualization with annotated cutoffs and a results table

**Cutoff interpretations:**
- *Zero crossings*: where the model switches from pushing toward sensitive (negative SHAP) to pushing toward escape (positive SHAP) or vice versa. These are the cleanest "regime boundary" cutoffs.
- *Rate-of-change peaks*: where the SHAP curve transitions most rapidly. These mark the centers of regime transitions.
- *Inflection points* (second derivative sign change): where curvature changes — useful for identifying boundaries between distinct regimes.

**Default analysis**: `relativePTClocation` (the U-shape feature). Change `FEATURE` in the config cell to analyze other features.

In [ ]:
# ============================================================================
# SETUP
# ============================================================================
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.interpolate import UnivariateSpline
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Configuration

Edit this cell to change which feature is analyzed.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# --- Input ---
SHAP_DATA_PATH = "../results/figures/visualizations_cv/shap_manuscript/shap_values_for_sharing.pkl"

# --- Feature to analyze ---
FEATURE = 'relativePTClocation'   # Primary feature for cutoff analysis
STRATIFY_BY = "last.EJC"                # Optional: e.g., 'last.EJC' or 'log2_CDS' for stratified analysis (None = no stratification)

# --- Smoother settings ---
LOESS_FRAC = 0.10                 # LOESS bandwidth: smaller = more wiggly, larger = smoother
GRID_POINTS = 500                 # Number of points on which to evaluate the smoother

# --- Bootstrap settings ---
N_BOOTSTRAP = 500                 # Number of bootstrap iterations for CIs
BOOTSTRAP_SEED = 42
CI_LEVEL = 95                     # Confidence level (percent)

# --- Cutoff detection ---
PEAK_PROMINENCE_FRAC = 0.30       # Rate-of-change peaks must be at least this fraction of the max |derivative|. Higher = fewer, more dominant peaks.
INCLUDE_INFLECTIONS = False       # Inflections (curvature changes) are usually LOESS wiggle artifacts; keep off unless specifically needed.

# --- Output ---
SAVE_PLOTS = True
SAVE_DIR =   "../results/figures/visualizations_cv/shap_manuscript/cutoff_analysis"
RESULTS_TSV = "../results/figures/visualizations_cv/shap_manuscript/cutoff_analysis/cutoffs_relativePTClocation.tsv"                # Optional path to save the results table; set to e.g. "cutoffs_relativePTClocation.tsv"

if SAVE_PLOTS:
    os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Will analyze: {FEATURE}")
if STRATIFY_BY:
    print(f"Stratified by: {STRATIFY_BY}")
else:
    print("No stratification (analyzing pooled SHAP values)")

## Load SHAP data

In [ ]:
# ============================================================================
# LOAD
# ============================================================================
with open(SHAP_DATA_PATH, 'rb') as f:
    shap_data = pickle.load(f)

shap_values_oof = shap_data['shap_values']
X_shap_oof = shap_data['feature_data']
categorical_color_maps = shap_data.get('categorical_color_maps', {})
feature_display_names = shap_data.get('feature_display_names', {})
base_value_avg = shap_data.get('base_value', 0.0)

print(f"SHAP values shape: {shap_values_oof.shape}")
print(f"Feature data shape: {X_shap_oof.shape}")
print(f"Base value: {base_value_avg:.4f}")

# Sanity check: feature must exist
if FEATURE not in X_shap_oof.columns:
    raise KeyError(f"Feature '{FEATURE}' not in data. Available: {list(X_shap_oof.columns)[:20]}...")
if STRATIFY_BY and STRATIFY_BY not in X_shap_oof.columns:
    raise KeyError(f"Stratification feature '{STRATIFY_BY}' not in data")

feat_idx = X_shap_oof.columns.get_loc(FEATURE)
shap_vals = shap_values_oof[:, feat_idx]
feature_vals = pd.to_numeric(X_shap_oof[FEATURE], errors='coerce').values

# Drop rows with NaN feature values
valid = ~np.isnan(feature_vals)
shap_vals = shap_vals[valid]
feature_vals = feature_vals[valid]
if STRATIFY_BY:
    stratify_vals = X_shap_oof[STRATIFY_BY].values[valid]
else:
    stratify_vals = None

print(f"\nAfter dropping NaN: {len(shap_vals)} variants")
print(f"Feature range: [{np.min(feature_vals):.3f}, {np.max(feature_vals):.3f}]")
print(f"SHAP range:    [{np.min(shap_vals):.3f}, {np.max(shap_vals):.3f}]")

## Helper functions

These do the actual work: LOESS fit, zero-crossing detection, rate-of-change peaks, and bootstrap CIs.

In [ ]:
# ============================================================================
# HELPERS
# ============================================================================

def fit_loess(x, y, frac=0.10, grid_points=500):
    """Fit LOESS smoother and return (grid, smoothed_values)."""
    # statsmodels lowess returns sorted (x, y_smooth) at the input x's
    smoothed = sm.nonparametric.lowess(y, x, frac=frac, return_sorted=True)
    sx, sy = smoothed[:, 0], smoothed[:, 1]
    # Re-evaluate on a regular grid via interpolation
    grid = np.linspace(np.min(x), np.max(x), grid_points)
    grid_y = np.interp(grid, sx, sy)
    return grid, grid_y


def find_zero_crossings(grid, smoothed):
    """Find feature values at which the smoother crosses zero.
    Returns list of (location, direction) where direction is 'pos_to_neg' or 'neg_to_pos'.
    Uses linear interpolation between adjacent grid points.
    """
    crossings = []
    for i in range(len(smoothed) - 1):
        y1, y2 = smoothed[i], smoothed[i + 1]
        if y1 * y2 < 0:  # sign change between adjacent grid points
            x1, x2 = grid[i], grid[i + 1]
            # Linear interpolation to find exact crossing location
            zero_x = x1 - y1 * (x2 - x1) / (y2 - y1)
            direction = 'pos_to_neg' if y1 > 0 else 'neg_to_pos'
            crossings.append((zero_x, direction))
        elif y1 == 0 and y2 != 0:
            # Exact zero hit (rare with continuous smoothers but possible)
            direction = 'pos_to_neg' if y2 < 0 else 'neg_to_pos'
            crossings.append((grid[i], direction))
    return crossings


def find_rate_of_change_peaks(grid, smoothed, prominence_frac=0.10):
    """Find local maxima of |derivative| of the smoother.
    These mark the steepest transitions in the SHAP relationship.
    Returns list of (location, derivative_value, direction).
    """
    dy = np.gradient(smoothed, grid)
    abs_dy = np.abs(dy)
    if np.max(abs_dy) == 0:
        return []
    peaks, _ = find_peaks(abs_dy, prominence=prominence_frac * np.max(abs_dy))
    out = []
    for p in peaks:
        direction = 'increasing' if dy[p] > 0 else 'decreasing'
        out.append((grid[p], dy[p], direction))
    return out


def find_inflection_points(grid, smoothed):
    """Find feature values where the second derivative changes sign.
    These mark transitions between curvature regimes (e.g., descending wing → trough).
    """
    dy = np.gradient(smoothed, grid)
    d2y = np.gradient(dy, grid)
    inflections = []
    for i in range(len(d2y) - 1):
        if d2y[i] * d2y[i + 1] < 0:
            inflections.append(grid[i])
    return inflections


def bootstrap_zero_crossings(x, y, frac, grid_points, n_boot, seed, ci_level=95):
    """Bootstrap the LOESS fit and re-extract zero crossings on each iteration.
    Returns: (point_estimate_crossings, dict mapping crossing_idx -> CI tuple)
    Bootstrap crossings are matched to point-estimate crossings by nearest location.
    """
    rng = np.random.default_rng(seed)
    n = len(x)

    # Point estimate from full data
    grid, smoothed = fit_loess(x, y, frac=frac, grid_points=grid_points)
    point_crossings = find_zero_crossings(grid, smoothed)

    # Bootstrap: resample with replacement and re-extract
    boot_crossings_per_point = [[] for _ in point_crossings]
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        try:
            g_b, s_b = fit_loess(x[idx], y[idx], frac=frac, grid_points=grid_points)
            boot_cross = find_zero_crossings(g_b, s_b)
        except Exception:
            continue
        # Match each bootstrap crossing to the nearest point-estimate crossing of the same direction
        for bx, bdir in boot_cross:
            same_dir = [(i, abs(px - bx)) for i, (px, pdir) in enumerate(point_crossings) if pdir == bdir]
            if same_dir:
                i_match = min(same_dir, key=lambda t: t[1])[0]
                boot_crossings_per_point[i_match].append(bx)

    # CIs
    alpha = (100 - ci_level) / 2
    cis = {}
    for i, samples in enumerate(boot_crossings_per_point):
        if len(samples) >= 10:
            lo = np.percentile(samples, alpha)
            hi = np.percentile(samples, 100 - alpha)
            cis[i] = (lo, hi, len(samples))
        else:
            cis[i] = (np.nan, np.nan, len(samples))

    return point_crossings, cis, grid, smoothed


print("Helpers defined.")

## Run cutoff analysis (pooled, no stratification)

This is the main analysis: fit the smoother, extract zero crossings with CIs, find rate-of-change peaks, and identify inflection points.

In [ ]:
# ============================================================================
# POOLED ANALYSIS
# ============================================================================
print(f"Running cutoff analysis on '{FEATURE}' (n = {len(shap_vals)})...")
print(f"  LOESS frac: {LOESS_FRAC}, grid: {GRID_POINTS}, bootstrap: {N_BOOTSTRAP}, CI: {CI_LEVEL}%")

point_crossings, cis, grid, smoothed = bootstrap_zero_crossings(
    feature_vals, shap_vals,
    frac=LOESS_FRAC, grid_points=GRID_POINTS,
    n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED, ci_level=CI_LEVEL
)
rate_peaks = find_rate_of_change_peaks(grid, smoothed, prominence_frac=PEAK_PROMINENCE_FRAC)
inflections = find_inflection_points(grid, smoothed) if INCLUDE_INFLECTIONS else []

# Build results table
rows = []
for i, (loc, direction) in enumerate(point_crossings):
    lo, hi, n_boot_used = cis.get(i, (np.nan, np.nan, 0))
    rows.append({
        'cutoff_type': 'zero_crossing',
        'direction': direction,
        'feature_value': loc,
        'ci_low': lo,
        'ci_high': hi,
        'n_bootstrap_samples': n_boot_used,
        'derivative_value': np.nan,
    })
for loc, dval, direction in rate_peaks:
    rows.append({
        'cutoff_type': 'rate_of_change_peak',
        'direction': direction,
        'feature_value': loc,
        'ci_low': np.nan,
        'ci_high': np.nan,
        'n_bootstrap_samples': np.nan,
        'derivative_value': dval,
    })
for loc in inflections:
    rows.append({
        'cutoff_type': 'inflection',
        'direction': 'curvature_change',
        'feature_value': loc,
        'ci_low': np.nan,
        'ci_high': np.nan,
        'n_bootstrap_samples': np.nan,
        'derivative_value': np.nan,
    })

results = pd.DataFrame(rows)

# Pretty print
print("\n=== ZERO CROSSINGS ===")
if point_crossings:
    for i, (loc, direction) in enumerate(point_crossings):
        lo, hi, n_b = cis.get(i, (np.nan, np.nan, 0))
        ci_str = f"[{lo:.4f}, {hi:.4f}]" if not np.isnan(lo) else "(insufficient bootstrap samples)"
        print(f"  {direction:12s}  at {FEATURE} = {loc:.4f}   {CI_LEVEL}% CI {ci_str}   (n_boot = {n_b})")
else:
    print("  None detected.")

print("\n=== RATE-OF-CHANGE PEAKS ===")
if rate_peaks:
    for loc, dval, direction in rate_peaks:
        print(f"  {direction:12s}  at {FEATURE} = {loc:.4f}   d(SHAP)/d({FEATURE}) = {dval:+.4f}")
else:
    print("  None detected.")

if INCLUDE_INFLECTIONS:
    print("\n=== INFLECTION POINTS ===")
    if inflections:
        for loc in inflections:
            print(f"  curvature change at {FEATURE} = {loc:.4f}")
    else:
        print("  None detected.")

if RESULTS_TSV:
    results.to_csv(RESULTS_TSV, sep='\t', index=False)
    print(f"\nResults saved to {RESULTS_TSV}")

results

## Visualize cutoffs

Three-panel plot:
- Top: SHAP scatter with LOESS fit and zero crossings (with bootstrap CIs as shaded bars)
- Middle: derivative (rate of change) with peaks marked
- Bottom: histogram of feature values to show data density at each cutoff

In [ ]:
# ============================================================================
# PLOT
# ============================================================================
display_name = feature_display_names.get(FEATURE, FEATURE)

fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1.5, 1]})
fig.patch.set_facecolor('white')

# --- Panel 1: SHAP scatter + LOESS + crossings ---
ax = axes[0]
ax.scatter(feature_vals, shap_vals, c='#888888', alpha=0.25, s=12,
           edgecolors='none', label='Variants')
ax.plot(grid, smoothed, color='#cc3333', linewidth=2.5, label=f'LOESS (frac={LOESS_FRAC})')
ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.6)

# Extra headroom at the top so crossing labels never collide with the frame
y0, y1 = ax.get_ylim()
ax.set_ylim(y0, y1 + 0.16 * (y1 - y0))

# Mark zero crossings with CI bars. Labels sit in a fixed band near the top
# of the (now-expanded) axes, staggered vertically so two nearby crossings
# don't overlap, each in a white box so it stays legible over the scatter.
for i, (loc, direction) in enumerate(point_crossings):
    lo, hi, _ = cis.get(i, (np.nan, np.nan, 0))
    color = '#1f77b4' if direction == 'pos_to_neg' else '#2ca02c'
    ax.axvline(loc, color=color, linewidth=2, alpha=0.85)
    if not np.isnan(lo):
        ax.axvspan(lo, hi, color=color, alpha=0.18)

    # Alternate label side (left/right of the line) and stagger vertically
    # by crossing index, so labels never sit on top of their own line.
    ha = 'left' if direction == 'pos_to_neg' else 'right'
    dx = 8 if ha == 'left' else -8
    y_frac = 0.97 - 0.11 * i
    ax.annotate(
        f'{direction}\n{loc:.2f}',
        xy=(loc, 1.0), xycoords=('data', 'axes fraction'),
        xytext=(dx, 0), textcoords='offset points',
        ha=ha, va='top', fontsize=11, fontweight='bold', color=color,
        bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=color, lw=1, alpha=0.9),
    )

ax.set_ylabel('SHAP value', fontsize=12, fontweight='bold')
ax.set_title(f'SHAP cutoff analysis: {display_name}',
             fontsize=14, fontweight='bold', pad=12)
ax.legend(loc='upper right', framealpha=0.95)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

# --- Panel 2: derivative ---
ax = axes[1]
dy = np.gradient(smoothed, grid)
ax.plot(grid, dy, color='#9467bd', linewidth=2)
ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.6)

# Headroom above the highest peak marker so its label never clips the frame
y0, y1 = ax.get_ylim()
ax.set_ylim(y0, y1 + 0.12 * (y1 - y0))

for loc, dval, direction in rate_peaks:
    color = '#d62728' if direction == 'increasing' else '#1f77b4'
    ax.axvline(loc, color=color, linewidth=1.5, alpha=0.8, linestyle=':')
    ax.scatter([loc], [dval], color=color, s=70, zorder=5, edgecolors='black', linewidths=1)
    # Push the label off to the side of the marker (not straight above it,
    # which is where the vertical line and marker already are) with a
    # white box so it reads cleanly against the curve and gridlines.
    side = 1 if dval >= 0 else -1
    ax.annotate(
        f'{loc:.2f}',
        xy=(loc, dval), xytext=(18 * side, 10 * (1 if dval >= 0 else -1)),
        textcoords='offset points', ha='left' if side > 0 else 'right',
        va='bottom' if dval >= 0 else 'top',
        fontsize=11, fontweight='bold', color=color,
        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, lw=1, alpha=0.9),
    )
ax.set_ylabel('d(SHAP)/d(feature)', fontsize=11, fontweight='bold')
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

# --- Panel 3: histogram of feature values ---
ax = axes[2]
ax.hist(feature_vals, bins=60, color='#aaaaaa', edgecolor='white', alpha=0.85)
for loc, direction in point_crossings:
    color = '#1f77b4' if direction == 'pos_to_neg' else '#2ca02c'
    ax.axvline(loc, color=color, linewidth=1.5, alpha=0.8)
ax.set_xlabel(display_name, fontsize=12, fontweight='bold')
ax.set_ylabel('Variants', fontsize=11, fontweight='bold')
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.tight_layout()

if SAVE_PLOTS:
    safe_name = FEATURE.replace('.', '_').replace('/', '_').replace(' ', '_')
    out_path = os.path.join(SAVE_DIR, f"cutoff_{safe_name}.png")
    plt.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f"Saved plot: {out_path}")

plt.show()

## Stratified analysis (optional)

If `STRATIFY_BY` is set, this runs the cutoff analysis separately for each stratum (e.g., separately for upstream-EJC, penultimate.last50bp, and last.exon variants).

This is useful for testing whether the cutoffs are consistent across categories. For categorical stratifiers each unique value gets its own analysis. For continuous stratifiers the data are split into tertiles.

In [ ]:
# ============================================================================
# STRATIFIED ANALYSIS
# ============================================================================
if STRATIFY_BY is None:
    print("STRATIFY_BY is None — skipping stratified analysis.")
    stratified_results = None
else:
    # Determine strata
    strat_series = pd.Series(stratify_vals)
    if strat_series.dtype.kind in 'OUS' or STRATIFY_BY in categorical_color_maps:
        # Categorical
        strata = [(v, strat_series == v) for v in strat_series.dropna().unique()]
    else:
        # Continuous → tertile split
        numeric = pd.to_numeric(strat_series, errors='coerce')
        q1, q2 = np.nanpercentile(numeric, [33.33, 66.67])
        strata = [
            (f'low (≤{q1:.2f})',    numeric <= q1),
            (f'mid ({q1:.2f}–{q2:.2f})', (numeric > q1) & (numeric <= q2)),
            (f'high (>{q2:.2f})',   numeric > q2),
        ]

    print(f"Running stratified analysis: {FEATURE} by {STRATIFY_BY}")
    print(f"  Number of strata: {len(strata)}\n")

    fig, axes = plt.subplots(len(strata), 1, figsize=(11, 4 * len(strata)), sharex=True)
    if len(strata) == 1:
        axes = [axes]
    fig.patch.set_facecolor('white')

    stratified_rows = []
    for ax, (name, mask) in zip(axes, strata):
        mask = np.asarray(mask)
        x_s = feature_vals[mask]
        y_s = shap_vals[mask]
        n_s = len(x_s)
        if n_s < 50:
            ax.text(0.5, 0.5, f'{name}: n={n_s} (too few for analysis)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=12)
            ax.set_title(f'{name} (n={n_s})', fontsize=11, fontweight='bold')
            continue

        try:
            pc, cis_s, g_s, sm_s = bootstrap_zero_crossings(
                x_s, y_s, frac=LOESS_FRAC, grid_points=GRID_POINTS,
                n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED, ci_level=CI_LEVEL
            )
        except Exception as e:
            ax.text(0.5, 0.5, f'{name}: fit failed ({e})',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            continue

        ax.scatter(x_s, y_s, c='#888888', alpha=0.3, s=10, edgecolors='none')
        ax.plot(g_s, sm_s, color='#cc3333', linewidth=2.2)
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)

        for i, (loc, direction) in enumerate(pc):
            lo, hi, n_b = cis_s.get(i, (np.nan, np.nan, 0))
            color = '#1f77b4' if direction == 'pos_to_neg' else '#2ca02c'
            ax.axvline(loc, color=color, linewidth=1.8, alpha=0.85)
            if not np.isnan(lo):
                ax.axvspan(lo, hi, color=color, alpha=0.18)
            ci_str = f"[{lo:.3f}, {hi:.3f}]" if not np.isnan(lo) else "(too few)"
            stratified_rows.append({
                'stratum': name, 'n': n_s,
                'cutoff_type': 'zero_crossing', 'direction': direction,
                'feature_value': loc, 'ci_low': lo, 'ci_high': hi,
                'n_bootstrap_samples': n_b,
            })
            print(f"  {name:30s}  {direction:12s}  {loc:.4f}  {CI_LEVEL}% CI {ci_str}")

        ax.set_title(f'{name} (n={n_s})', fontsize=11, fontweight='bold')
        ax.set_ylabel('SHAP value', fontsize=10, fontweight='bold')
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_linewidth(1.0)

    axes[-1].set_xlabel(display_name, fontsize=12, fontweight='bold')
    fig.suptitle(f'Stratified cutoff analysis: {display_name} by {STRATIFY_BY}',
                 fontsize=13, fontweight='bold', y=1.0)
    plt.tight_layout()

    if SAVE_PLOTS:
        safe_name = FEATURE.replace('.', '_').replace('/', '_').replace(' ', '_')
        safe_strat = STRATIFY_BY.replace('.', '_').replace('/', '_').replace(' ', '_')
        out_path = os.path.join(SAVE_DIR, f"cutoff_{safe_name}_by_{safe_strat}.png")
        plt.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f"\nSaved stratified plot: {out_path}")

    plt.show()

    stratified_results = pd.DataFrame(stratified_rows)
    if RESULTS_TSV and len(stratified_rows) > 0:
        strat_path = RESULTS_TSV.replace('.tsv', f'_stratified_by_{STRATIFY_BY}.tsv')
        stratified_results.to_csv(strat_path, sep='\t', index=False)
        print(f"Stratified results saved to {strat_path}")

stratified_results